<a href="https://colab.research.google.com/github/Yanem-G/ML_projects/blob/master/Customer_churn_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split,KFold
import tensorflow as tf
from tensorflow.keras import Sequential,models,layers
from tensorflow.keras.layers import Dense,Input
from tensorflow.keras.losses import binary_crossentropy
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.metrics import AUC
from sklearn.utils import class_weight
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score,classification_report

In [ ]:
df = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

df.head()

,Customer_ID,Age,Gender,Location,Subscription_Type,Account_Age_Months,Monthly_Spending,Total_Usage_Hours,Support_Calls,Late_Payments,Streaming_Usage,Discount_Used,Satisfaction_Score,Last_Interaction_Type,Complaint_Tickets,Promo_Opted_In,Churn
0,1001,19,Male,Illinois,Basic,50,152.44,416,5,2,61,76,3,Neutral,0,1,1
1,1002,41,Male,California,Premium,14,113.34,36,5,1,17,90,5,Negative,3,0,0
2,1003,44,Female,Florida,Basic,2,168.39,207,3,1,85,12,6,Neutral,3,0,1
3,1004,21,Male,Florida,Basic,55,197.12,379,4,3,54,32,4,Positive,3,1,0
4,1005,65,Male,New York,Premium,12,84.46,475,5,4,82,62,1,Neutral,0,0,1


In [ ]:
# drop the noise features found in the EDA

df = df.drop(["Gender","Location","Age","Last_Interaction_Type","Satisfaction_Score","Streaming_Usage"],axis=1)
test = test.drop(["Gender","Location","Age","Last_Interaction_Type","Satisfaction_Score","Streaming_Usage"],axis=1)

In [ ]:
# feature engineering : using signal features to create meaningful features

df["usage_rate"] = df["Total_Usage_Hours"]/df["Account_Age_Months"]
df["support_calls_rate"] = df["Support_Calls"]/df["Account_Age_Months"]
df["Spending_per_hour"] = df["Monthly_Spending"]/df["Total_Usage_Hours"]

test["usage_rate"] = test["Total_Usage_Hours"]/test["Account_Age_Months"]
test["support_calls_rate"] = test["Support_Calls"]/test["Account_Age_Months"]
test["Spending_per_hour"] = test["Monthly_Spending"]/test["Total_Usage_Hours"]

# using the new features with the original ones
df["Support_effecency"] = df["Support_Calls"]/(df["Complaint_Tickets"] + 1)

test["Support_effecency"] = test["Support_Calls"]/(test["Complaint_Tickets"] + 1)


In [ ]:
#one hot encoding

test["Subscription_Type"] = test["Subscription_Type"].replace({"Basic" : 0,"Premium" : 1,"Enterprise" : 2})
df["Subscription_Type"] = df["Subscription_Type"].replace({"Basic" : 0,"Premium" : 1,"Enterprise" : 2})

/tmp/ipython-input-1533413093.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test["Subscription_Type"] = test["Subscription_Type"].replace({"Basic" : 0,"Premium" : 1,"Enterprise" : 2})
/tmp/ipython-input-1533413093.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["Subscription_Type"] = df["Subscription_Type"].replace({"Basic" : 0,"Premium" : 1,"Enterprise" : 2})


In [ ]:
x = df.drop(["Churn"],axis=1)
y = df["Churn"]
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2)

In [ ]:
x_train.shape

(6400, 14)

In [ ]:
#we will randomize the hyper parameters
options={
    "num_of_layers":[3,4],
    "num_of_units":[32,64,128],
}

def select_hyper_par(options):
    num_of_units=[]
    num_of_layers = random.choice(options["num_of_layers"])
    for i in range(num_of_layers):
        num_of_units.append(random.choice(options["num_of_units"]))

    return{
        "num_of_layers":num_of_layers,
        "num_of_units":num_of_units,
    }
#build model
def build_model(options):
    model = Sequential()
    model.add(Input(shape=(14,)))
    for units in options["num_of_units"]:
        model.add(Dense(units=units , activation='relu'))
    model.add(Dense(units=1 , activation='sigmoid'))
    return model

In [ ]:
#define number of fold for k-CV
k=5
kf=KFold(n_splits=k,shuffle=True,random_state=42)

In [ ]:
# checking if the output data are imbalance
print(y_train.value_counts(normalize=True))

Churn
0    0.682031
1    0.317969
Name: proportion, dtype: float64


In [ ]:
# balance the weight of the output prediction, to prevent the model from choosing only churn = 0 and ignoring churn = 1 because they are fewer than the first
weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weight_dict = {0: weights[0], 1: weights[1]}
def Random_Search(n_iteration):
    best_score=0.5
    for i in range(n_iteration):
        average_auc=[]
        param=select_hyper_par(options)
        for train_idx,val_idx in kf.split(x_train):
            #train and val are arrays we can use them in indexing in numpy !!!
            x_cv_train, x_cv_test= x_train.iloc[train_idx], x_train.iloc[val_idx]
            y_cv_train, y_cv_test= y_train.iloc[train_idx], y_train.iloc[val_idx]
            #scaling cross validation data
            scaler = StandardScaler()
            x_cv_train = scaler.fit_transform(x_cv_train)
            x_cv_test= scaler.transform(x_cv_test)

            model = build_model(param)
            model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),loss = binary_crossentropy,metrics=['accuracy',AUC(name='auc')])
            early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
            model.fit(x_cv_train,y_cv_train,epochs=10,validation_data=(x_cv_test,y_cv_test),class_weight=class_weight_dict,callbacks=[early_stop],batch_size=32,verbose=0)
            # results = [loss, accuracy, auc]
            results = model.evaluate(x_cv_test,y_cv_test,verbose=0)
            #AUC measures how well your model can distinguish between class 0 and class 1. A score of 0.5 is a random guess, and 1.0 is a perfect model.
            auc = results[2]
            average_auc.append(auc)

        current_score=np.mean(average_auc)
        if current_score>best_score:
            best_score=current_score
            best_params=param
    return best_params,best_score
best_params,best_score = Random_Search(20)
print(f"best average accuracy:{best_score}")
print(f"best hyperparameters is {best_params}")


best average accuracy:0.5118492186069489
best hyperparameters is {'num_of_layers': 4, 'num_of_units': [32, 64, 32, 64]}


best average accuracy:0.5125334799289704
best hyperparameters is {'num_of_layers': 5, 'num_of_units': [32, 128, 64, 128, 128]}

best average accuracy:0.5133551478385925
best hyperparameters is {'num_of_layers': 3, 'num_of_units': [32, 128, 64]}

best average accuracy:0.5141546130180359
best hyperparameters is {'num_of_layers': 3, 'num_of_units': [128, 64, 32]}

best average accuracy:0.5168764114379882
best hyperparameters is {'num_of_layers': 3, 'num_of_units': [64, 64, 32]}

In [ ]:
# final scaling
final_scaler = StandardScaler()
x_train = final_scaler.fit_transform(x_train)
x_test = final_scaler.transform(x_test)

In [ ]:

#we're going to use early stopping to see when the accuracy start decreasing , it help to know how many epoch do I need
early_stop = EarlyStopping(monitor='val_auc', patience=10, restore_best_weights=True)
best_model = build_model(best_params)
best_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),loss = binary_crossentropy,metrics=['accuracy',AUC(name='auc')])
best_model.fit(x_train,y_train,epochs=200,callbacks=[early_stop],batch_size=32)


Epoch 1/200
200/200 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step - accuracy: 0.6741 - auc: 0.4883 - loss: 0.6391
Epoch 2/200
 63/200 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6828 - auc: 0.5535 - loss: 0.6217

/usr/local/lib/python3.12/dist-packages/keras/src/callbacks/early_stopping.py:153: UserWarning: Early stopping conditioned on metric `val_auc` which is not available. Available metrics are: accuracy,auc,loss
  current = self.get_monitor_value(logs)


200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6787 - auc: 0.5395 - loss: 0.6261
Epoch 3/200
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6748 - auc: 0.5599 - loss: 0.6263
Epoch 4/200
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6781 - auc: 0.5627 - loss: 0.6228
Epoch 5/200
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.6833 - auc: 0.5809 - loss: 0.6166
Epoch 6/200
200/200 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.6812 - auc: 0.5951 - loss: 0.6154
Epoch 7/200
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6823 - auc: 0.6100 - loss: 0.6107
Epoch 8/200
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6769 - auc: 0.6171 - loss: 0.6119
Epoch 9/200
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.6768 - auc: 0.6236 - loss: 0.6100
Epoch 10/200
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6856 - auc: 0.6408 - loss: 0.5989
Epoch 11/200
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6897 - auc: 0.6646 - loss: 0.5924


In [ ]:
probabilities = best_model.predict(x_test)
predictions = (probabilities > 0.5).astype(int)

50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [ ]:
# Evaluate the model
r2 = r2_score(y_test, predictions)
print(classification_report(y_test,predictions))
print(f'R^2 Score: {r2}')

              precision    recall  f1-score   support

           0       0.71      0.68      0.70      1130
           1       0.31      0.35      0.33       470

    accuracy                           0.58      1600
   macro avg       0.51      0.51      0.51      1600
weighted avg       0.60      0.58      0.59      1600

R^2 Score: -1.0214648842025977
